# UK Exports Predictor

A model that learns from global HS6 goods trade, 2012–2024, to estimate the UK's **Export Potential** for every Product and Market. Export Potential minus Actual Exports is the **Export Gap**, and positive gaps ranked by size are **Opportunities**. Re-predicting with only the trade-policy features changed gives the **Scenario Effect** of a Trade Agreement, and the pairs it would unlock are **Blocked Opportunities**. A FastAPI service serves the saved model to government export Advisors through a Streamlit app.

**How to read this notebook.** It is an argument, in sections that each say what they show and end with a **Conclusion**. Every table and chart is read from a saved artefact through the `exportspredictor` package, which the API imports too, so training and serving compute features identically; the notebook defines no functions and, on real data, trains and fits nothing. Each modelling choice names the course concept it applies or extends (*Course:* …, from `ML_AI_Concept_Notes.html`). Results are associations on held-out data, never causes or forecasts. Terms follow `GLOSSARY.md`, and decisions that look odd are explained in `docs/adr/`.

## Parameters

`FIXTURE_MODE` is the notebook's one parameter (the cell is tagged `parameters`).

- `FIXTURE_MODE = True` (the default) runs every section on a tiny synthetic world in seconds. That world has no saved artefact, so the notebook builds one through the pipeline. The test suite runs the notebook this way (`tests/test_notebook.py`); cells that need the real downloads say so and skip.
- `FIXTURE_MODE = False` (set by `EXPORTSPREDICTOR_FIXTURE_MODE=0`, as `scripts/rebuild.py` does) runs on the real data and **reads; it does not build**. The models are trained, evaluated and saved by `python -m exportspredictor.pipeline`, and the PPML benchmark is fitted once by `python -m exportspredictor.ppml`. The notebook loads the newest artefact, the experiment loop's saved results and the derived tables, so a run cannot change a reported number. It takes about two minutes.

In [ ]:
import os

FIXTURE_MODE = True  # True: the tiny fixture world, in seconds. False: the real data (reads the newest artefact and saved results; trains and builds nothing).

if os.environ.get("EXPORTSPREDICTOR_FIXTURE_MODE") == "0":
    FIXTURE_MODE = False  # scripts/rebuild.py sets this for the real run
FIXTURE_MODE

In [ ]:
import polars as pl

from exportspredictor import (
    artefacts,
    baci,
    evaluation,
    experiments,
    loop,
    model_choice,
    pipeline,
    policy,
    reporting,
)
from exportspredictor.evaluation import NOT_RUN
from exportspredictor.fixture import build_world
from exportspredictor.paths import artefacts_dir

In [ ]:
if FIXTURE_MODE:
    world = build_world()
    # The fixture's ablation and learning curve travel with its artefact, as the real ones do.
    experiment_results = experiments.run(experiments.fixture_data(world), experiments.FIXTURE_PLAN)
    bundle = pipeline.build_artefacts(
        world, source="fixture", experiment_results=experiment_results, parts=pipeline.Parts.every()
    )
else:
    # Trained, evaluated and saved by `python -m exportspredictor.pipeline` (several hours on an RTX 3080);
    # the newest version is read here, with the saved ablation and learning curve. Nothing is rebuilt.
    bundle = artefacts.load_latest(artefacts_dir())
    experiment_results = experiments.load()
report = bundle.evaluation
print(reporting.provenance(bundle))

## 1. The question and the headline

**The question.** Export Advisors must decide which UK industries to help, and in which Markets. They mostly work from what the UK already sells, which hides Markets where the UK sells far less of a Product than comparable Exporters do. For each Product, the model ranks the Markets by UK Export Potential, so that an Advisor knows where to look first.

**How it is judged.** The measure is the one an Advisor reads: for a typical Product, how closely the model's order of Markets matches what the UK actually exported, in a year no model saw. It is the rank correlation within each Product, averaged over Products (1 is a perfect order, 0 no better than chance). The bar was set before the test year was read: beat the **Naive Baseline**, a market-share rule with no model (the UK's share of the Product's world exports times each Market's imports of it), which already ranks UK Markets well (*Course:* baselines first: a model is worth what it adds over the simplest real alternative).

The headline below is printed from the artefact's evaluation report, then drawn beside the Naive Baseline and the two earlier versions.

In [ ]:
print(report["earlier_versions"]["text"])

In [ ]:
reporting.ranking_by_version(bundle)

**Conclusion.** On 2023, read once, the served model ranks UK Markets more closely to what the UK exported than the Naive Baseline does: the success bar stated in advance is met. v2, the version before it, fell below the Naive Baseline, and v1 sat between the two. The margin is modest, and it is an association on one test year, not a promise of what any Market will buy. Sections 2 to 4 describe the data and the model, section 5 why v2 fell short, section 6 how the model was re-chosen, and section 7 the result and what it cost. Sections 8 to 11 cover Scenarios as Illustrative figures, explanations, the limits and governance, and the conclusions.

## 2. The data: Flows, Zero Cells and policy features

Everything the model knows comes from public trade and country data. This section shows what the target is, how absent trade enters training, and where the policy features come from (*Course:* exploratory data analysis and the data-quality audit, at a scale where the data never sit fully in memory: compact Parquet scanned lazily with Polars).

- **Flows**: CEPII's BACI, HS 2012 revision, release V202601 (ADR 0003). Every non-zero Flow between about 226 Exporters and Importers for about 5,200 Products, 2012–2024, about 142 M in all. BACI reconciles each country's export report with its partners' import reports, so a UK Flow is not HMRC's figure. 2024 is provisional: late reporters are still missing.
- **Gravity and size**: distance, contiguity, language and colonial ties from CEPII Gravity; GDP and population from the World Bank, with the IMF and Gravity filling gaps under one written rule that records each value's source.
- **Policy features**: Trade Agreement, Customs Union and EU, CPTPP and EFTA membership per pair and year, from the WTO's RTA database with per-member dates (`docs/research/trade-agreement-coverage.md`), and the Applied Tariff from UNCTAD TRAINS, carried forward from 2021 and zero under an agreement (ADR 0006). These are the only columns a Scenario changes.

Flows per year, with world value and the UK's share of it:

In [ ]:
if FIXTURE_MODE:
    flows = build_world().flows.lazy()
else:
    # Downloaded and converted by the BACI build step (`baci.prepare`, in scripts/rebuild.py); read here.
    flows = baci.scan_flows()
flows.head(5).collect()

In [ ]:
flow_summary = baci.summarise_flows(flows)
flow_summary

**Skew.** Flow values are extremely skewed: the mean UK Flow is about a hundred times its median. That is why the model's loss is Tweedie rather than squared error on raw values (*Course:* descriptive statistics: a skewed variable is described by its median, and one huge value can rule a squared loss; section 4).

In [ ]:
uk_latest = flows.filter((pl.col("exporter").cast(pl.String) == baci.UK) & (pl.col("value_usd") > 0))
uk_latest.group_by("year").agg(
    mean_usd=pl.col("value_usd").mean(),
    median_usd=pl.col("value_usd").median(),
    markets=pl.col("importer").n_unique(),
    products=pl.col("hs6").n_unique(),
).sort("year").collect()

**Zero Cells.** BACI records only non-zero Flows, so an absent pair means zero *or* unreported. A model of Export Potential must also learn where trade does not happen, but the full Exporter × Importer × Product × year grid has about 3.45 bn cells. Training therefore adds **Zero Cells**: the Exporter exports the Product to someone and the Importer imports it from someone, but not from each other. Every UK Zero Cell is kept, because New Market and New Product Opportunities are UK Zero Cells. Other Exporters' are sampled at 5% and weighted by 20, so weighted totals stay unbiased (the Horvitz–Thompson estimator; *Course:* class imbalance, handled by subsampling the majority and reweighting it, so the loss still sees zeros in their true proportion).

Training rows per year, the Zero Cells sampled and the relevant zeros they stand for:

In [ ]:
from exportspredictor import features

if FIXTURE_MODE:
    feature_world = build_world()
    feature_table = features.build_features(
        features.training_rows(feature_world.flows, rate=0.5), feature_world.flows, feature_world.sources
    ).lazy()
else:
    feature_dir = features.features_dir()  # written by `python -m exportspredictor.features`; read here, not rebuilt
    feature_table = pl.scan_parquet(feature_dir / "training_*.parquet")

is_zero = pl.col("value_usd") == 0
feature_table.group_by("year").agg(
    rows=pl.len(),
    sampled_zeros=is_zero.sum(),
    relevant_zeros_estimated=pl.col("weight").filter(is_zero).sum(),
).sort("year").collect()

**Data quality.** Every Exporter and Importer maps to an ISO3 code with Gravity features (the first cell below, real data only). Before trusting BACI as the target, its UK Flows were checked against HMRC's own 2023 Overseas Trade Statistics, an independent primary source (*Course:* the data-quality audit, taken beyond the notes). HMRC records about a sixth more than BACI, almost all for known reasons: value HMRC does not attribute to a Product or Market, and non-monetary gold. Pairs that both sources record agree closely in rank (0.89), so BACI is a sound target for the *shape* of UK trade, which is what the ranking uses (`docs/research/baci-hs12-feasibility.md`).

In [ ]:
from exportspredictor import gravity
from exportspredictor.paths import data_dir

if not FIXTURE_MODE:
    # Built by the gravity step (`python -m exportspredictor.gravity`, in scripts/rebuild.py); read here, not rebuilt.
    derived = data_dir() / "derived"
    in_flows = pl.concat(
        [flows.select(baci_iso3=pl.col(side).cast(pl.String)) for side in ["exporter", "importer"]]
    ).unique().collect()
    # Only countries that appear in the 2012-2024 Flows matter; BACI's file also lists retired codes.
    country_map = pl.read_parquet(derived / "country_map.parquet").join(in_flows, on="baci_iso3", how="semi")
    country_year = pl.read_parquet(derived / "country_year.parquet").filter(pl.col("iso3").is_in(country_map["iso3"].implode()))
    print("Exporters and Importers in the Flows:", country_map["baci_iso3"].n_unique())
    print("... with no Gravity pair features:", gravity.unresolved(country_map).height)
    print("... with no WDI data:", country_map.filter(~pl.col("in_wdi")).height)
    display(country_map.filter((pl.col("baci_iso3") != pl.col("iso3")) | pl.col("gravity_proxy").is_not_null()))

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the HMRC cross-check needs the real downloads (FIXTURE_MODE = False).")
else:
    from exportspredictor import hmrc

    # The allocated OTS table is written by `hmrc.build` (in scripts/rebuild.py); read here, not rebuilt.
    hmrc_check = hmrc.crosscheck(
        hmrc.load(), hmrc.baci_uk_exports(flows, country_map)
    )
    print(hmrc_check["text"])
    display(pl.DataFrame([{key: value for key, value in hmrc_check.items() if key.endswith("_usd")}]))
    display(pl.DataFrame(hmrc_check["by_sector"]))
    display(pl.DataFrame(hmrc_check["by_market"]))
    print(hmrc_check["pairs"])

**Policy features.** A membership counts in a year if it began before 1 July, Gravity's rule: EU–Japan counts from 2019, UK–Australia and UK–New Zealand from 2023, and the UK's CPTPP accession (December 2024) only from 2025. Checked against CEPII Gravity for 2012–2021, the two sources agree on every agreement the Scenarios and Event Backtests rely on, and where they differ the reasons are known, mostly gaps in Gravity (*Course:* feature engineering with data lineage: every flag traces to a dated primary record). The UK's Trade Agreements for a few Markets, then the cross-check:

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the fixture world carries its own trade_agreement column (section 4).")
else:
    policy_files = policy.policy_paths()  # written by `python -m exportspredictor.policy`; read here, not rebuilt
    agreements = pl.read_parquet(policy_files["agreements"])
    trading_blocs = pl.read_parquet(policy_files["trading_blocs"])
    print(agreements.height, "pair-years with a Trade Agreement;", trading_blocs.height, "country-years in a tracked Trading Bloc")

In [ ]:
if not FIXTURE_MODE:
    uk_rows = pl.DataFrame(
        {"exporter": "GBR", "importer": ["FRA", "JPN", "AUS", "NZL", "MEX", "USA"]}
    ).join(pl.DataFrame({"year": list(range(2018, 2025))}, schema={"year": pl.Int32}), how="cross")
    uk_policy = policy.attach_policy(uk_rows, agreements, trading_blocs)
    display(uk_policy.pivot(on="year", index="importer", values="trade_agreement"))
    display(uk_policy.pivot(on="year", index="importer", values="customs_union"))
    display(uk_policy.filter(pl.col("importer") == "FRA").select("year", "eu_o", "cptpp_o", "efta_o"))

In [ ]:
if not FIXTURE_MODE:
    gravity_policy = policy.read_gravity_policy()  # from the gravity step's extract (section 2)
    crosscheck, disagreements = policy.gravity_crosscheck(agreements, trading_blocs, gravity_policy)
    display(crosscheck)

**Conclusion.** BACI gives a consistent HS 2012 panel of about 142 M Flows, small enough to scan lazily on one machine, and close to HMRC on the shape of UK trade. Three of its properties shape later choices: values are extremely skewed (a Tweedie loss), absent Flows mean zero or unreported (sampled, reweighted Zero Cells, and weaker rankings where reporting is poor), and 2024 is provisional (a 2023 test year). The policy features come from dated primary records, and they bound what any Scenario Effect can mean.

## 3. Features and the leakage rules

Every model input is computed once, in `exportspredictor.features`, which training and the API both import, so a feature cannot drift between them (*Course:* training/serving skew, removed by construction). This section lists the features, then states the three rules that keep the evaluation honest, each with its reason.

| Group | Features | What it captures |
|---|---|---|
| Gravity | log GDP and population of both countries, log distance, contiguity, common language, colonial tie, common colonizer | size and closeness of the two countries |
| Supply | `log_supply_loo`, `rca_loo`: the Exporter's exports of the Product to *other* Importers, and its revealed comparative advantage | what the Exporter makes and sells well |
| Demand | `log_demand_loo`, `high_income_share_loo`: the Importer's imports of the Product from *other* Exporters, and the share from high-income ones | what the Market buys, and from whom |
| Product | `sector` (HS2 chapter), `relatedness` (closeness to what the Exporter already exports competitively) | Sector differences and nearby capabilities |
| Re-export | `reexport_ratio`: the Importer's exports of the Product against its imports | Re-export Hubs (ADR 0005) |
| Policy | Trade Agreement, Customs Union, EU, CPTPP and EFTA flags; `log_applied_tariff` | the only columns a Scenario changes (ADR 0006) |

### The leakage rules

1. **Out-of-fold scoring** (ADR 0001). UK Product–Market pairs are split into five folds by a hash of the pair, so all of a pair's years share a fold, and each UK Export Potential comes from the fold model that never saw that pair. *Why*: a model that had seen the pair would copy its Actual Exports back and shrink its Export Gap towards zero (*Course:* out-of-fold target encoding, applied to the whole model rather than to one feature).
2. **Leave-one-out supply features.** Every aggregate a row feeds leaves out the row's own Flow, so UK supply of a Product excludes the target Market. *Why*: otherwise the feature contains the answer (*Course:* target leakage: use only what exists without the target). The pair's trade in other Products is left out too, because it would absorb the policy effect. One gap is accepted and recorded in ADR 0001: a non-UK row's demand still includes the UK's Flow to that Market.
3. **The time split.** Fit on 2012–2021, choose on 2022, test once on 2023; clip ranges, the Sector encoder and every aggregate are fitted on training years only. *Why*: a choice or a preprocessing step that saw the test year makes its score optimistic, silently (*Course:* preprocessing and evaluation leakage; train, validation and test splits, split by time when predicting later years).

Tests enforce each rule (`tests/test_features.py`, `tests/test_folds.py`): changing a held-out pair's Flow moves none of its own features, and distorting one fold's UK labels a thousandfold leaves that fold's scores unchanged. One UK row's features, whisky (`220830`) to Japan:

In [ ]:
feature_table.filter(
    (pl.col("exporter") == "GBR") & (pl.col("importer") == "JPN") & (pl.col("hs6").cast(pl.String) == "220830")
).sort("year").tail(1).collect().transpose(include_header=True, column_names=["value"])

**Conclusion.** One function builds every input for training and serving alike. Three tested rules mean that no UK Export Potential is computed by a model that saw its pair, from a feature that contains its own Flow, or under a choice that saw the test year. The one accepted leak, the UK inside other rows' demand, is small and recorded in ADR 0001.

## 4. The model and how it is scored out-of-fold

The model is one gradient-boosted tree ensemble per fold, trained with XGBoost on the GPU, and the served Export Potential is a **Blend** of it with the Naive Baseline (section 6 says how that form was chosen). This section says what each part does and how a UK pair is scored.

- **Gradient boosting** (*Course:* ensembles: boosting adds small trees that each correct the errors of the last). Depth-12 trees and a learning rate of 0.1, v2's configuration.
- **Tweedie loss** (*Course:* loss functions; choosing a loss for a skewed target). Flows are non-negative, often zero and very skewed. The Tweedie objective models a log mean whose variance grows with the mean, so a few huge Flows do not rule the fit.
- **Monotone constraints** on the policy features: +1 on every membership and −1 on the Applied Tariff, which is clipped to its training range (ADR 0006). Adding a Trade Agreement never lowers Export Potential; ADR 0008 kept the constraints after testing them against the ranking (section 6).
- **Five fold models** (*Course:* k-fold cross-validation, used here for cross-fitting rather than only for a score). Fold model *k* trains on every non-UK row plus the UK pairs outside fold *k*, and scores only fold *k*.
- **Memory** (*Course:* GPUs and device placement). The 162 M training rows of 2012–2023 do not fit the RTX 3080's 10 GB, so they stream year by year into a `QuantileDMatrix`, and the largest hash sample of non-UK rows that fits is kept and reweighted (ADR 0004). Every UK row is kept.

The served configuration, read from the artefact:

In [ ]:
{key: bundle.metadata.get(key) for key in ("model_form", "params", "rounds", "n_folds", "non_uk_sampling_rate", "training_seconds")}

### From Export Potential to the Export Gap

The fold models, refit on 2012–2023, score the UK grid of each **Gap Period** year, 2022 to 2024, each pair by the fold model that held it out. The **Export Gap** is the mean of the three yearly gaps (Export Potential minus Actual Exports, a year with no Flow counting as $0). Averaging damps the year-to-year noise of single shipments (*Course:* bagging averages to cut variance). 2020 and 2021 are left out because COVID and the end of the Brexit transition describe shocks, not fundamentals. The Baseline table holds the gaps, and the 2024 Export Potential that Scenarios start from:

In [ ]:
baseline = bundle.baseline.join(bundle.products, on="hs6", how="left").join(
    bundle.markets.rename({"name": "market"}), on="iso3", how="left"
)
baseline.select(
    pairs=pl.len(),
    products=pl.col("hs6").n_unique(),
    markets=pl.col("iso3").n_unique(),
    pairs_with_exports=(pl.col("actual_exports_usd") > 0).sum(),
    actual_exports_usd=pl.col("actual_exports_usd").sum(),
    export_potential_usd=pl.col("export_potential_usd").sum(),
    baseline_export_potential_usd=pl.col("baseline_export_potential_usd").sum(),
    opportunities=(pl.col("export_gap_usd") > 0).sum(),
)

How one pair's Export Gap is formed, year by year: whisky (`220830`) to Japan.

In [ ]:
whisky_japan = (pl.col("hs6") == "220830") & (pl.col("iso3") == "JPN")
display(bundle.history.filter(whisky_japan, pl.col("export_gap_usd").is_not_null()))
bundle.baseline.filter(whisky_japan).select(
    "export_potential_usd", "actual_exports_usd", "export_gap_usd", "baseline_export_potential_usd"
)

**Conclusion.** Every UK Export Potential comes from a fold model that held its pair out, under a loss suited to skewed, zero-heavy Flows and constraints that fix the direction of every policy feature. Every Export Gap averages three recent years. Sections 5 and 6 turn to what the model does well and what the Blend adds.

## 5. What v2 showed

v2, the version before this one, was the model alone, chosen step by step on Tweedie deviance. On 2023 it ranked UK Markets below the Naive Baseline. This section shows two reasons: the measure it was chosen on moved against the ranking, and it was weak on whether the UK exports at all.

### Every choice was made on deviance

Early stopping, the feature ablation, the learning curve and the hyperparameter search (Bayesian, TPE with a median pruner) all chose on the 2022 Tweedie deviance, with the UK ranking only as a guardrail (ADR 0002). The guardrail passed by a hair, −0.0099 against a limit of −0.01. Among the three configurations the search confirmed on every Product, the winner had the lowest deviance and the lowest UK ranking (*Course:* loss functions vs evaluation metrics: training optimises the loss it is given, and the metric people read can disagree with it). The search's confirmed configurations, from the artefact (details in the appendix):

In [ ]:
pl.DataFrame(bundle.metadata["search"]["confirmation"]).select("trial", "rounds", "validation_deviance", "uk_spearman")

### Weaker on whether the UK exports at all

Most UK pairs are Zero Cells, so a Product's order of Markets is decided largely by which Markets have any UK trade. On the 2022 validation fold, v2's configuration ranks Markets the UK already serves better than the Naive Baseline does (`non_zero_spearman`), but it ranks "any Flow against none" worse (`zero_vs_non_zero_spearman`). Market share knows something about where trade happens that the model lacks. From the experiment loop's record:

In [ ]:
record = bundle.metadata["ranking_experiments"]
measures = ["selection_measure", "non_zero_spearman", "zero_vs_non_zero_spearman", "tweedie_deviance"]
pl.DataFrame([
    {"model": f"v2's configuration ({record['noise_floor']['arms'][0]['arm']})", **{m: record["noise_floor"]["arms"][0].get(m) for m in measures}},
    {"model": "Naive Baseline", **{m: record["naive"].get(m) for m in measures}},
])

On 2023, under a comparison with v1 fixed before the test year was read, v2 estimated the size of Flows better than v1 but ranked UK Markets less well. The gain ladder associates most of that drop with v2's new training rows rather than its features or tuning. Both are frozen in the appendix.

**Conclusion.** v2 was a good model of the size of trade: its deviance over all Exporters was the lowest of any version. It was chosen on that, and the ranking an Advisor reads fell below the Naive Baseline. Two lessons carry into section 6: choose on the measure that is reported, and use what the Naive Baseline knows about whether trade happens at all.

## 6. Re-choosing the model on the Selection Measure

The model was re-chosen by an experiment loop (`exportspredictor.loop`, ADR 0008) under a rule fixed in code before any experiment was read. This section explains the rule, then shows every Arm the loop ran, failures included, the five-fold confirmation and the Blend it chose. Everything here is on the 2022 validation year; the loop's data refuse 2023 (`experiments.TestYearRead`). Full detail: `docs/research/opportunity-ranking-experiments.md`.

### The rule

- **Selection Measure**: section 1's headline measure, moved to the validation year: the within-Product rank correlation for held-out UK pairs in 2022, averaged over Products. Every other metric, Tweedie deviance included, is reported and never chosen on. Rounds are early-stopped on it too.
- **Noise Floor**: v2's configuration trained with three seeds, which change both the row sample and XGBoost's seed. The spread of the Selection Measure across seeds is how far it moves by chance (*Course:* seeds, run-to-run noise and the noise band: a gap smaller than the spread between runs is noise, not evidence).
- **Win Threshold**: twice the Noise Floor's standard deviation, and at least 0.005. An Arm is kept only if it beats the best so far by more than this; otherwise every Arm of that experiment is dropped (*Course:* defensible selection and the one-standard-error rule, made stricter and fixed in advance).
- Arms run on one seed against a cached base, so comparisons are paired. No Arm was added or re-run after its result was seen.

In [ ]:
summary = model_choice.ranking_summary(bundle.metadata.get("ranking_experiments"))
print(model_choice.how_chosen(bundle.metadata, report)["text"])
None if summary is None else {key: summary[key] for key in ("validation_year", "noise_floor_sd", "win_threshold")}

### Every Arm

Seven experiments, in order: blend with the Naive Baseline or start from it; select configurations on ranking (v1's, three from v2's search, shallower trees); a two-part model (whether trade happens, then how much); two Zero Cell scopes; three Product-detail features; and the monotone constraints off. The table lists every Arm with its Selection Measure, its change on the best so far, its best round and whether it was kept; the chart draws each change against the Win Threshold. The first cell also checks that the artefact records the loop's saved results.

In [ ]:
ranking_results = bundle.metadata["ranking_experiments"] if FIXTURE_MODE else loop.ranking_record(loop.load())
print("The artefact records the experiment loop's saved results:", ranking_results == bundle.metadata["ranking_experiments"])
with pl.Config(tbl_rows=20):
    display(reporting.experiments(bundle))

In [ ]:
reporting.experiments_chart(bundle)

**The stopping-rule caveat.** Four Arms reached their best round within their first 50 and then stopped: v1's configuration, depth 6, the starting point and the two-part model's binary part (the table shows its size part's rounds). Rounds stop on the model alone, while Arms are chosen on the Blend, so for these four the result reflects the stopping rule as much as the idea; with more patience, or stopping on the Blend, they might have scored higher. The rule was kept, because changing it after seeing results would have meant adding Arms mid-run. A dropped Arm here is not a disproved idea. The one fully trained alternative configuration (search trial 23) still landed inside the noise.

**What the Arms say.** Only the Blend cleared the Win Threshold. Every later Arm landed within about +0.003 of the Blend or below it: the configuration barely matters once blended, the Zero Cell scopes and Product-detail features moved nothing beyond the noise, and removing the monotone constraints ranked slightly worse, so they stay (ADR 0008).

### The five-fold confirmation

A one-fold win could be the luck of the fold. So the Blend and v2 (the same fold model, alone) were trained on all five folds and measured on 2022, each fold model scoring only the UK pairs it held out (*Course:* k-fold cross-validation: a score and its spread across folds). Fold by fold, then over every UK pair of 2022 (`all`):

In [ ]:
reporting.confirmation(bundle)

The Blend beats v2 on every fold, each time by several times the Win Threshold, and over every pair it also beats the Naive Baseline, which v2 alone does not.

### The Blend

$$\text{Export Potential} = \exp\big(0.7\,\log \text{model} + 0.3\,\log \text{Naive Baseline}\big),$$

each side floored at $1, with the Naive Baseline's weight, 0.3, chosen on the Selection Measure. The two parts know different things. The Naive Baseline orders Markets by whether trade is plausible at all, which is where v2 was weak; the model estimates size among the Markets the UK serves, from gravity, demand, Relatedness and policy. A weighted geometric mean keeps Export Potential in dollars, so the Export Gap is still defined. The Naive Baseline has no policy features, so a Scenario moves only the model's share: every Scenario Effect is the model's, scaled by 0.7 on the log scale (*Course:* ensembles, here two members with a fixed weight chosen as a hyperparameter on validation data).

In [ ]:
print(bundle.model_form.text)

**Conclusion.** Under a rule fixed before any result was read, seven experiments kept one change: the Blend with the Naive Baseline. It beat v2 on all five folds of 2022, by about five times the Win Threshold overall. Everything else, including four Arms that the stopping rule cut short, stayed inside the noise or below it, and is reported rather than hidden.

## 7. The 2023 result and what it cost

The final run trained the served Blend's fold models and read 2023 once, after the confirmation (*Course:* train, validation and test splits: look at the test set once). This section reads that result beside v2, v1 and the Naive Baseline, then states what choosing on the ranking cost and the one deviation from the rule.

The test-year fold models train on 2012–2022 and score 2023, each UK pair by the fold model that held it out, as the Baseline is scored. 2023 shaped no choice: the search, the loop and the confirmation never read it, and a test multiplies every 2023 Flow by 1,000 and checks that the search results do not move.

### The result, read once

Every version's UK ranking and costs, from the evaluation report. Two Naive Baseline rows are shown because their Flows differ slightly (every Exporter's, as the Blend uses, or v2's training rows); they agree closely, so the verdict does not depend on which.

In [ ]:
versions = report["earlier_versions"]
pl.from_dicts(versions["rows"] + versions["naive"], infer_schema_length=None)

Every metric, model against Naive Baseline, for UK pairs scored out-of-fold and for every Exporter's 2023 rows (*Course:* regression metrics, reported beside the ranking measure rather than chosen on):

In [ ]:
pl.DataFrame([{"scope": scope, **result["overall"]} for scope, result in report["accuracy"].items()]).drop("group", strict=False)

### What choosing on the ranking cost

ADR 0008 expected the size of Export Potential to get worse, and it did. The cost, printed from the report:

In [ ]:
traded = model_choice.traded_away(report.get("earlier_versions"))
print("No earlier version of this artefact records a lower Tweedie deviance." if traded is None else traded["text"])

**Lift.** Lift asks whether large Export Gaps point at growth. The past is replayed on Exporters held out like the UK: gaps over 2015–2017, then whether each pair's 2023 Flow rose by at least 50% and $100,000. Raw Lift, the top tenth's growth rate over all pairs', is well above 1, but large gaps sit on large pairs, so the **size-matched Lift** compares the top tenth with pairs of similar Export Potential. It is below 1, as it was for v2 and v1: an Opportunity marks a Market worth a look, not one likelier to grow than its peers of the same size. Lift is a tendency, not a forecast (*Course:* lift as a gate number, here with a cluster bootstrap over Exporters and a size-matched control).

In [ ]:
lift = report.get("lift")
print(f"Lift: {NOT_RUN}." if lift is None else lift["text"])
None if lift is None else pl.DataFrame(lift["by_decile"]).rename({"decile": "Export Gap tenth (1 = largest)", "rate": "share that grew", "pairs": "weighted pairs"})

In [ ]:
reporting.lift_by_tenth(bundle)

### The one deviation: the served round count

The rule says deviance chooses nothing. The served fold models break it in one place: they trained for v2's round count, which the hyperparameter search early-stopped on Tweedie deviance, scaled to the fold models' rows. The code review found this after 2023 was read. Rebuilding would read 2023 a second time, so the project owner chose not to rebuild. The figures above are those of the served models, and the next rebuild takes its rounds from the Selection Measure (the median of the five-fold confirmation's rounds, then scaled). From the artefact:

In [ ]:
reporting.round_count(bundle)

**Conclusion.** Chosen on the Selection Measure, the Blend beats the Naive Baseline on 2023, the bar set in advance, where v2 did not. The gain cost accuracy on size: Tweedie deviance over all Exporters rose by about a fifth on v2's, and the size-matched Lift stays below 1. So the ranking is the part of the output to trust most, and dollar values deserve more caution. These are associations on one test year, read once, with one disclosed deviation from the rule.

## 8. Scenarios and Blocked Opportunities as Illustrative figures

A **Scenario** asks the served model a what-if question: copy the 2024 Baseline rows of the affected Markets, change **only the policy columns** (Trade Agreement, Customs Union, the Trading Bloc flags and the Applied Tariff that goes with them, ADR 0006), and re-predict each pair through `prediction.export_potential` with the fold model that scored its Baseline (ADR 0001). The **Scenario Effect** is the difference. The Naive Baseline has no policy features, so a Scenario moves only the model's share of the Blend. `exportspredictor.scenarios` checks on every run that no other column moved, so a Scenario cannot report a change in demand as a policy effect.

This section shows the Trading Bloc presets and the Blocked Opportunity values, then sets them beside two benchmarks: a PPML gravity regression and the Event Backtests. That comparison is why every Scenario Effect and Blocked Opportunity value is labelled **Illustrative**: an upper-end association, never a forecast, and never chosen on (ADR 0008). The package's Scenario note, the text the API serves beside every Scenario, comes first (*Course:* the counterfactual test, flip one attribute and hold the rest, used here as a measurement rather than a fairness check; and correlation is not causation: the answer is still learned from correlations in past trade).

In [ ]:
from exportspredictor import illustrative, scenarios

note = illustrative.scenario_note(bundle)
print(f"{note['label']}. {note['text']}")
for part in ("benchmark", "model_association", "event_backtest"):
    print("-", note[part]["text"])
if note["blend"] is not None:
    print("-", note["blend"]["text"])

### The Trading Bloc presets

Four presets change only the rows of a bloc's member Markets in the Baseline year. **EU re-join** sets the Trade Agreement, Customs Union and EU flags, as for any pair of EU members in training; **EU Customs Union only** sets the Customs Union; **CPTPP** sets the Trade Agreement and CPTPP flags, with accession treated as complete; **Leave** switches off the UK's membership of every bloc it is in. Joining sets a known Applied Tariff to 0 where no agreement existed and leaves a missing one missing; removing an agreement restores the 2021 MFN rate (ADR 0006).

A membership counts in a year only if it began before 1 July, so the 2024 Baseline has no UK CPTPP membership, and the CPTPP preset is a live prediction that 2025 data can check. In 2024 the UK is in no tracked Trading Bloc, so Leave changes nothing. Each preset's Scenario Effect, in dollars and as a share of its Markets' Baseline Export Potential, with its Blocked Opportunities:

In [ ]:
preset_results = {
    name: scenarios.run(bundle.baseline_features, bundle.baseline, bundle.fold_models, scenarios.Scenario(preset=name), form=bundle.model_form)
    for name in scenarios.PRESETS
}
pl.DataFrame(
    [
        {
            "preset": scenarios.PRESETS[name].title,
            "pairs_changed": result.pairs.height,
            **result.total(),
            "blocked_opportunities": result.blocked_opportunities().height,
        }
        for name, result in preset_results.items()
    ]
)

On the served artefact, EU re-join adds about $38bn, 28% of its Markets' Baseline Export Potential, and CPTPP about $4.7bn, 12%. The EU figures are large because the model associates intra-EU membership with much denser trade than an agreement alone brings; they describe what the model has learned from EU members' trade, not what re-joining would deliver. CPTPP is smaller because the UK already has agreements with 9 of its 11 members, so for those only the CPTPP flag changes, while Brunei and Malaysia also gain an agreement and a zero tariff and show the largest shares. Per Market:

In [ ]:
preset_results["cptpp"].by_market().join(bundle.markets.select("iso3", "name"), on="iso3", how="left")

### Blocked Opportunity values

For every UK pair with no Trade Agreement in 2024, the **Blocked Opportunity value** is the Scenario Effect of adding one, computed when the artefact is built (`scenarios.blocked_opportunity_values`) through the same prediction function; it is 0 where an agreement already exists. A pair is a **Blocked Opportunity** when that value is at least 20% of its Baseline Export Potential and at least $1M. The app can rank by it, always under the Illustrative label. The largest values are aero-engines, cars and medicines to the USA, China and India: large Markets with which the UK has no agreement. The heading, the total and the CSV export in the app all carry the word "illustrative", so the label travels with the figure when an Advisor copies it into a briefing.

In [ ]:
baseline.with_columns(
    blocked_opportunity=scenarios.is_blocked("blocked_opportunity_value_usd", "baseline_export_potential_usd")
).sort("blocked_opportunity_value_usd", descending=True).select(
    "hs6", "description", "market", "export_potential_usd", "blocked_opportunity_value_usd", "blocked_opportunity"
).head(10)

### Beside the PPML benchmark

The monotone constraint fixes the *direction* of a Scenario Effect: adding an agreement never lowers Export Potential. Its *size* is only what the trees learned, so it is set beside the method trade economists use: a structural gravity regression fitted by Poisson pseudo-maximum likelihood (PPML, `exportspredictor.ppml`), fitted once and saved with the artefact. Flows are summed to Sector level for 2012, 2015, 2018, 2021 and 2024, three years apart so that trade can adjust:

$$\text{Flow}_{e,i,s,t} = \exp\big(a_{e,s,t} + b_{i,s,t} + c_{e,i,s} + \beta\,\text{TradeAgreement}_{e,i,t}\big)\cdot\varepsilon.$$

The pair effects $c_{e,i,s}$ absorb everything fixed about a pair, including the fact that countries that sign agreements already trade a lot, so $\beta$ is identified only by pairs whose status changed, each compared with itself; $e^{\beta}-1$ is the change the regression associates with an agreement. It is a benchmark only: its pair effects need a pair's own Flows, which a held-out UK pair does not have (ADR 0001), so it could never serve Export Potential (*Course:* linear regression and reading its coefficients, taken further: a log link and Poisson loss so that zeros stay in, fixed effects as a very large one-hot encoding swept out by alternating projections, and standard errors clustered by pair). The model's side is the Blocked Opportunity Scenario recomputed for every UK pair without an agreement through the served Blend. The published estimates are listed with their sources in `docs/research/rta-effects.md`.

In [ ]:
reporting.ppml_benchmark(bundle)

Three readings, all associational.

- **The benchmark is small**: about +5%, with a 95% interval that only just includes zero, below the whole published range (+21% to +100%). The reasons are known: BACI has no intra-national Flows, which more than double the estimate in the WTO/UNCTAD *Advanced Guide*; the agreement dummy is contemporaneous; and many agreements join partners that already trade on low tariffs.
- **The served model's association is about seven times larger**: +37% weighted by Baseline Export Potential over 678,216 UK pairs (+45% as a plain mean per pair). It lies inside the published range but far outside the benchmark's interval on the same Flows and years. v2, the model alone, gave +57%; the Blend's is smaller because only its model share moves.
- **Why the model sits high.** The trees learn the agreement association *across* pairs: pairs with agreements trade more, and the other features cannot absorb everything that makes those pairs close. The benchmark compares each pair with itself; the model cannot, because a held-out UK pair has no pair effect. Part of what the model attributes to an agreement is the closeness of the countries that sign one, the endogeneity Baier and Bergstrand (2007) warn about.

### Beside the Event Backtests

A benchmark is still a model. An **Event Backtest** checks a Scenario Effect against an agreement that really happened (`exportspredictor.backtests`). The primary one is the **EU–Japan Economic Partnership Agreement**, in force from 1 February 2019:

1. Train one model on 2012–2018 only. A test multiplies every Flow from 2019 on by 1,000 and checks that the trained model does not change.
2. Predict EU–Japan Flows for 2019–2021 with the agreement on and off. The difference is the model's Scenario Effect.
3. Measure what happened with a **difference-in-differences**: the change in EU exports to Japan from 2016–2018 to each later year, against the change to ten **Control Markets**, large high-income Markets with no EU agreement in the window, with a 95% interval from resampling Products (*Course:* beyond the notes; it takes the counterfactual test from the model to the observed data).

COVID disrupted 2020 and 2021 unevenly, so 2019 is the cleanest year. A secondary backtest uses the UK–Australia and UK–New Zealand agreements (in force May 2023), scored by the test-year fold models, with provisional 2024 as its only post year. The Event Backtests measure the model alone, trained before the Blend was chosen; a served Scenario Effect is that effect scaled by 0.7 on the log scale.

In [ ]:
backtest = report.get("event_backtest")
print(f"Event Backtests: {NOT_RUN}." if backtest is None else backtest["text"])
reporting.event_backtests(bundle)

**Conclusion.** Scenarios change only policy columns and re-predict through the served Blend, and the monotone constraint makes every added agreement raise Export Potential. The direction is designed in; the size is not. After the EU–Japan agreement the model alone associated +35% with EU exports to Japan where +2% was observed (95% interval −7% to +14%), and +42% against about 0% in the other direction; the Blend would serve +23%, still outside the observed interval. For UK–Australia and New Zealand, +31% met an observed fall. The PPML benchmark on the same Flows associates an agreement with +5%, against the served model's +37%. So Scenario Effects and Blocked Opportunity values are **Illustrative**: they show where the model associates an agreement with most trade, and an upper-end order of magnitude, never a forecast of what an agreement will add. The app, the API and the model card show the same Scenario note beside every such figure.

## 9. Explaining Export Potential

An Advisor who takes an Opportunity to a firm must say *why* the model expects more trade. This section shows what drives Export Potential overall and why one pair scores as it does (*Course:* explainability, global against local, and SHAP for local explanations that are additive and faithful to the model you ship).

XGBoost computes exact TreeSHAP values itself, so no background sample is needed. Each pair is explained by **its own fold model**, the one that held it out and scored its Baseline (ADR 0001): any other model would explain a number the Advisor never sees. The Blend is explained as it is served: the model's SHAP values are scaled by its weight, 0.7, and the Naive Baseline's share is attributed half to UK supply and half to Market demand, its two inputs (`prediction.log_contributions`). Contributions add up on the log scale, so the base value plus every contribution reproduces the Export Potential exactly (`tests/test_explain.py`), and on the dollar scale each factor multiplies Export Potential by `exp(contribution)`.

**Global importance** is the mean absolute contribution over a random sample of 2,000 Baseline pairs. Unlike impurity importance, it is measured on the served predictions and cannot credit a feature the model never uses (*Course:* feature importance, impurity against permutation). A sample of 20,000 gave the same top five and moved no feature by more than about 2%, at ten times the run time.

In [ ]:
from exportspredictor import explain

importance = explain.global_importance(bundle, sample=2_000)  # about 80 s on the CPU with the depth-12 fold models
importance.with_columns(typical_multiplier=pl.col("mean_abs_contribution").exp())

On the served artefact, demand dominates: how much the Market imports the Product from others moves a typical pair by a factor of about 3.8, and UK supply of the Product to other Markets by about 3.0. Both carry the Naive Baseline's share as well as the model's. Then come the high-income share of the Market's imports (×1.5), distance (×1.4) and the Market's GDP (×1.4). The bloc flags move a typical pair by about 1%: most pairs' policy status never changes, so Scenario Effects rest on a small part of the model, one more reason to read them as Illustrative. Low importance is also a governance signal (*Course:* mean |SHAP| supports data minimisation): the common-colonizer feature adds about 1% and could be dropped at no cost, while the bloc flags stay because Scenarios need them.

**A local explanation.** The largest Opportunity is parts of turbo-jets and turbo-propellers (`841191`) to France. The Why? page shows the same factors in the same words:

In [ ]:
top_opportunity = baseline.sort("export_gap_usd", descending=True).row(0, named=True)
why = explain.explain_pair(bundle, top_opportunity["hs6"], top_opportunity["iso3"])
print(f"{top_opportunity['description'][:60]} to {why.market}: Export Potential {explain.usd(why.export_potential_usd)}, scored by fold model {why.fold}")
display(pl.DataFrame([{"factor": f.label, "value": f.value_text, "multiplier": f.multiplier} for f in why.factors[:8]]))
pl.DataFrame([{"year": point.year, "actual_exports_usd": point.actual_exports_usd} for point in why.history])

The served 2024 Export Potential for UK turbo-jet parts in France is about $1.4bn, from fold model 2, against Actual Exports near $0.1bn in every year since 2019. The factors read as an Advisor would want: the UK exports $12.5bn of this Product to other Markets, France imports $2.5bn of it from others, the UK is 8.6 times as specialised in it as the world average, and France is close. One factor is a warning: France *re-exports* 378% of what it imports of these parts. It hosts large aero-engine and aircraft makers, so much of this trade runs inside supply chains that existing contracts decide. The history adds what the model cannot see: Actual Exports have not moved while Export Potential rose with the provisional 2024 data.

Two limits apply to every explanation. SHAP values are **associations inside the model**: "distance lowers Export Potential" means comparable distant pairs trade less, not that distance causes it. And the largest factors (a Market's size, its imports, distance) are not things an Advisor can change; the actionable reading is the supply side, that the UK already sells this Product well elsewhere (*Course:* actionable explanations).

**Conclusion.** Every Export Potential decomposes exactly into named factors from the fold model that produced it, in words an Advisor can repeat. Globally the served model is driven by what Markets buy and what the UK sells, which is also what the Naive Baseline knows; policy features carry a small part of each prediction. A local explanation read beside the Actual Exports history is the Advisor's main defence against a misleading Opportunity.

## 10. Limits and governance

A result is only as useful as its stated limits. This section reads the known limitations, a sanity check of the top Opportunities and the finding that removed the Uncertain Opportunity badge. It then gives one short entry per governance topic of the course, each pointing to the artefact that is its evidence, and leaves the rest to the model card (*Course:* responsible AI as design constraints with owners and evidence: a commitment without an artefact is an aspiration).

### Known limitations

These are served by the API and shown on the app's Model Evaluation page, so that Advisors can caveat their briefings. Two weigh most: every Applied Tariff after 2021 is carried forward (ADR 0006), and sanctions, prohibitions and other non-tariff barriers are invisible to the model.

In [ ]:
with pl.Config(fmt_str_lengths=200, tbl_rows=20):
    display(pl.DataFrame(evaluation.LIMITATIONS))

### Sanity check: the top Opportunities

A model can score well and still rank nonsense, so the largest Export Gaps are read as a trade economist would read them. No Excluded Product may appear at all (ADR 0005). Flagged Products and Re-export Hubs stay in the rankings with a caveat, so the share of the top of the list they hold is measured, beside the share of Uncertain Opportunities used below. Then whisky (`220830`), a Product an Advisor knows well.

In [ ]:
from exportspredictor.exclusions import is_excluded

top_opportunities = baseline.filter(pl.col("export_gap_usd") > 0).sort("export_gap_usd", descending=True)
display(top_opportunities.select(
    "hs6", "description", "market", "opportunity_type", "export_potential_usd", "actual_exports_usd", "export_gap_usd",
).head(10))
pl.concat(
    [
        top_opportunities.head(top).select(
            top=pl.lit(top),
            excluded_products=is_excluded().sum(),
            flagged_product_share=pl.col("flagged_product").mean(),
            reexport_hub_share=pl.col("reexport_hub").mean(),
            uncertain_share=pl.col("uncertain_opportunity").mean(),
        )
        for top in (20, 100, 1000)
    ]
)

In [ ]:
baseline.filter(pl.col("hs6") == "220830").sort("export_gap_usd", descending=True).select(
    "market", "export_potential_usd", "actual_exports_usd", "export_gap_usd"
).head(8)

On the served artefact no Excluded Product appears. Flagged Products hold 5% of the top 20 and 2% of the top 1,000. Re-export Hubs hold about a fifth of the top 100 and of the top 1,000 (Hong Kong, Ireland, Belgium, the Netherlands), as expected for large importers of what the UK makes, which is why their rows carry a caveat. The top Opportunities are plausible: aero-engines and their parts (France, Hong Kong, China, Türkiye, Germany), packaged medicines (France, Italy, the USA) and engines and aircraft to the USA, all *Expand* pairs in the UK's known strengths. Whisky shows what the model cannot see: the **Russian Federation** ranks fourth, a Market where UK exports collapsed under sanctions from 2022, and no feature records sanctions. An Advisor should read such rows as Barriers the model does not know about, not as Opportunities.

### The Potential Range and the Uncertain Opportunity share

Each UK pair has a **Potential Range**: the range within which Actual Exports fall 80% of the time for pairs of about the same Export Potential (ADR 0007). It is split conformal prediction, calibrated on other Exporters' 2024 rows, never the UK's (a UK residual is its Export Gap with the sign flipped), within 20 bands of Export Potential (Mondrian bands, because Tweedie errors grow with the prediction), with coverage checked on a held-out half of the rows (*Course:* held-out validation, extended from one error figure for the model to a range for each prediction with a coverage guarantee). An Opportunity whose Actual Exports lie inside its range is an **Uncertain Opportunity**: its Export Gap is within the scatter normally seen among comparable pairs. The finding, printed from the evaluation report, then coverage by band:

In [ ]:
from exportspredictor import conformal

potential_range = report["potential_range"]
print(conformal.coverage_text(potential_range))
pl.DataFrame(potential_range["by_band"])

Coverage on the held-out half is 85% against the 80% target. The excess sits in the smallest bands, where most Actual Exports are exactly $0 and ties widen the range; from a few thousand dollars upwards every band is within about a point of 80%. By Exporter, the large, diversified Exporters most like the UK sit near the target, which supports carrying the range over to UK pairs, though that stays an assumption.

**88% of Opportunities are Uncertain Opportunities**, yet none of the top 1,000 by Export Gap is (the sanity check above). Most positive gaps are the small gaps of *New Market* pairs, well inside the ordinary scatter. A badge on nearly nine rows in ten separated nothing, so the badge and its filter were removed from the ranking pages. The share is reported here, on the Model Evaluation page and in the model card, and each pair's range stays on the Why? page (ADR 0007, Outcome). One limit remains: the offsets are dollar amounts within a band, so every pair in the top band gets the same width, about $24M, a sliver for a $1bn pair. A range relative to Export Potential would mean more for large pairs.

### Model card

The course asks for a short card covering intended and out-of-scope use, owner, metrics by group, limitations and data sources, generated from code and versioned with the model, printing UNASSIGNED rather than omitting a section. `governance.model_card` builds it from the artefact, so its numbers are the evaluated ones, and it leads as this notebook does: the headline, the Model Form, the Illustrative Scenario note, how the form was chosen and the Potential Range share, with the frozen results last. The pipeline writes it into each artefact folder; a real run reads it, and the fixture run writes its own. The owner is **UNASSIGNED** on purpose: naming who answers for the model is a decision for the team that adopts it.

In [ ]:
from IPython.display import Markdown

from exportspredictor import governance

# The fixture run saves its own artefact and card; the real ones were written by the pipeline and are only read here.
saved_to = artefacts.save(bundle, artefacts_dir()) if FIXTURE_MODE else artefacts_dir() / bundle.version
card_path = governance.write_model_card(bundle, saved_to) if FIXTURE_MODE else saved_to / governance.MODEL_CARD_FILE
Markdown(card_path.read_text(encoding="utf-8"))

### Data lineage

Every served number should trace back to exact inputs (*Course:* data lineage and reproducible artefacts). Raw downloads are fingerprinted with their size and SHA-256 (`governance.data_lineage`), including the 1,294 UNCTAD TRAINS files and the UN HS concordances. Every derived table is rebuilt by one command, in the order of `scripts/rebuild.py`. Each artefact is a folder named by its UTC timestamp, whose metadata records the BACI release, the Exporter scope, the sampling rates and seed, the parameters, the XGBoost version and the device, and `copied_from` for a version derived from another (*Course:* registry versions). The API serves the newest complete folder, so rolling back means removing it. One gap remains: the artefact does not yet store the raw-file hashes, so proving which download a version was trained on relies on the data folder not having changed.

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the fixture world is generated in code (exportspredictor.fixture), so there are no raw files to fingerprint.")
else:
    lineage = governance.data_lineage()  # hashes about 2.8 GB of downloads, under a minute
    display(lineage.group_by("source").agg(files=pl.len(), bytes=pl.col("bytes").sum()).sort("source"))
    display(lineage.filter(pl.col("source").is_in(["UNCTAD TRAINS", "UN HS concordances"])))
{
    **{key: bundle.metadata.get(key) for key in (
        "version", "source", "xgboost_version", "device", "zero_sampling_rate", "uk_zero_sampling_rate",
        "zero_sampling_seed", "non_uk_sampling_rate", "created_at", "copied_from",
    )},
    "exporters_in_scope": len(bundle.metadata.get("exporter_scope") or []),
}

### Bias review

Bias here concerns countries and Sectors, not people: the model holds no personal data. BACI is built from what countries report, and poorer countries report later and less completely, so absent Flows read as zeros make a poorly reporting Market look as if it buys little (data bias). One loss, dominated by large Flows, serves every Market (objective bias). The review is a representation audit and a worst-group evaluation on the UK's 2023 out-of-fold grid, by region and by World Bank income level as a stand-in for reporting quality; a group with fewer than 500 UK Flows is flagged thin (*Course:* representation audit, and evaluating beyond the average).

In [ ]:
uk_accuracy = report["accuracy"]["uk_out_of_fold"]
bias_columns = ["group", "rows", "non_zero_flows", "actual_usd", "model_spearman_within_product", "naive_spearman_within_product"]
for breakdown in ("by_region", "by_income"):
    if uk_accuracy.get(breakdown):
        display(pl.DataFrame(uk_accuracy[breakdown]).select(bias_columns).with_columns(thin=pl.col("non_zero_flows") < 500))
pl.DataFrame(uk_accuracy["by_sector"]).select(bias_columns).head(5)  # the weakest Sectors (HS chapters)

The gradient is steep: the Blend ranks UK Markets at 0.65 in high-income Markets and 0.27 in low-income ones, and by region from 0.82 (North America) to 0.35 (Latin America and the Caribbean). It is mostly a data effect: in low-income Markets only about one UK pair in ten has any trade, so the rankings are among near-zero Flows that BACI measures least well. The Blend beats the Naive Baseline in every group, and no group is thin. By Sector the weakest are commodities, such as ores, fish, live animals and wood pulp, where sanitary rules, quotas and logistics decide Markets and the model sees none of them. The cost falls on developing Markets: an Advisor reading the rankings at face value would see fewer, noisier Opportunities there, for reasons of data quality rather than fundamentals. The mitigations in place are the region breakdown on the Model Evaluation page and a region filter on every ranking; per-group thresholds (*Course:* per-group thresholds) would come next if the rankings were used to allocate budgets.

### Regulation

**EU AI Act.** Intended purpose decides the tier, not the technique (*Course:* the EU AI Act's risk tiers). The purpose here is decision support for public export promotion, from aggregate trade statistics about countries and Products, never natural persons. The system is not prohibited and not high-risk (no Annex III use, not a safety component), and it neither talks to people as if human nor generates synthetic content, so it is **minimal risk**, with no new legal duties. A UK government tool used in the UK falls outside the Act anyway, but classifying it is the right discipline. **Function creep would change the tier**: scoring individual firms for grants would be a decision about specific businesses, so the model card lists it as out of scope.

**The UK approach** has no AI statute: regulators apply five principles, and the course asks for an artefact per principle. Safety and robustness: the time split, the leakage tests and the Event Backtests. Transparency: the Why? page and the model card. Fairness: the bias review. Accountability: versioned artefacts, ADRs and the decision log, with the owner still UNASSIGNED. Contestability: outputs are leads that Advisors decide on, each with its explanation. The data are country-level aggregates, so UK GDPR is not engaged.

### Drift

A trained model is frozen while trade moves (*Course:* model drift, its three types, and the Population Stability Index). **Data drift** is checked with each feature's PSI on the UK grid, 2023 against 2024: stable below 0.10, act above 0.20. **Concept drift**, the same fundamentals bringing different trade after new tariffs or sanctions, shows only as falling accuracy, so it needs the yearly re-evaluation. **Label drift** comes from BACI revising its provisional year. Then the stability of the rankings between the two years, scored by the same models:

In [ ]:
if FIXTURE_MODE:
    last_year = feature_table.select(pl.col("year").max()).collect().item()
    drift_reference = feature_table.filter(pl.col("year") == last_year - 1).collect()
    drift_current = feature_table.filter(pl.col("year") == last_year).collect()
else:
    drift_reference = pl.read_parquet(feature_dir / "uk_grid_2023.parquet")
    drift_current = pl.read_parquet(feature_dir / "uk_grid_2024.parquet")
display(governance.feature_drift(drift_reference, drift_current, features.FEATURES))
print(report["stability"]["text"])

Every Market-side feature is stable. Three read *act*, for known reasons: the UK's own GDP and population take one value per year on the UK grid, so any change moves every row (PSI is the wrong test for a constant, which the clip ranges fitted on training years guard instead), and Relatedness, a UK-side feature, moves with the provisional 2024 export basket and is the one to watch. The Applied Tariff cannot drift because it is carried forward, which is a limit of the data, not evidence of stability. Rankings are broadly stable between 2023 and 2024, but only about three in five of the top 1,000 Opportunities are the same, another reason the Export Gap averages the Gap Period.

**The yearly refresh** follows each January's BACI release: rebuild, check the inputs (the HMRC cross-check and PSI), retrain with the test year moved on, promote only if the Naive Baseline is beaten, and use the 2025 data as the first live test of the CPTPP Scenario.

**Conclusion.** The limits are measured and stated where the numbers are read. The top Opportunities are plausible, but sanctions and prohibitions are invisible to the model. 88% of Opportunities lie inside their Potential Range, so that share is reported as a finding instead of a badge that marked almost every row. The model ranks UK Markets much better where trade data are good, and Advisors are told so. Governance has an artefact for each course topic: a generated model card, fingerprinted inputs and versioned artefacts, a bias review, a minimal-risk classification with function creep ruled out of scope, and drift checks with a yearly refresh. The open gaps are named: no owner, raw-file hashes not stored in the artefact, and no per-region thresholds.

## 11. Conclusions

The question was whether a model trained on global trade can tell Advisors where to look first better than a market-share rule. This section says what worked, what did not, and what the project would do next.

**What worked**

- **The ranking beats the Naive Baseline on 2023**, read once: 0.555 against 0.523, where v2 (0.511) did not and v1 (0.540) did by less. The bar was set before the test year was read.
- **Choosing on the measure that is reported.** The Selection Measure, a Noise Floor from three seeds and a Win Threshold fixed in code turned seven experiments into one kept change, the Blend, confirmed on all five folds of 2022 (*Course:* defensible selection, made stricter and fixed in advance).
- **The integrity rules held.** Out-of-fold scoring, leave-one-out features and the time split are tested, so no reported UK figure comes from a model that saw its pair or a choice that saw the test year.
- **Honest labels.** Scenario Effects are shown as Illustrative beside a PPML benchmark and the Event Backtests, and the Uncertain Opportunity share replaced a badge that separated nothing.
- **Methods beyond the notes**: out-of-fold cross-fitting, Tweedie loss with Horvitz–Thompson weighted Zero Cells, monotone constraints, split conformal prediction with Mondrian bands, PPML with three-way fixed effects, difference-in-differences backtests, a pre-registered comparison with a paired bootstrap, and a size-matched Lift with a cluster bootstrap.

**What an Advisor gets.** For any Product, the Markets ranked by UK Export Potential, with the Export Gap that says where comparable Exporters sell more than the UK does. Each Opportunity comes with its explanation on the Why? page, its Actual Exports history and its Potential Range, so a lead can be checked before it is taken to a firm. The Scenarios and Blocked Opportunities pages answer "what does the model associate with an agreement here?", always beside the PPML benchmark and the Event Backtest result. The Model Evaluation page states the 2023 result, the cost in size, the Lift and the limitations in the words this notebook uses. What the Advisor does not get is a forecast: every figure is an association learned from global trade, and the ranking is the part to trust most.

**What did not**

- **v2's tuning.** Every v2 choice was made on Tweedie deviance, and the ranking Advisors read fell below the Naive Baseline (section 5). The search, the ablation and the learning curve are now frozen results.
- **Size got worse.** Chosen on the ranking, the Blend's deviance over all Exporters rose by about a fifth on v2's, and its size-matched Lift stays below 1 (0.76): an Opportunity marks a Market worth a look, not one likelier to grow than its peers of the same size.
- **Scenario Effects are too large**: +35% predicted against +2% observed after EU–Japan, and +37% against a +5% benchmark. The monotone constraint makes the direction consistent; it does not make the size right.
- **Four Arms were cut short** by a stopping rule on the model alone, so their ideas are untested rather than disproved.
- **One deviation from the rule**: the served round count came from v2's deviance search, and the next rebuild takes it from the Selection Measure.

**What next**, in order of value:

1. **Rebuild on the Selection Measure's round count**, then let the next BACI release's test year decide, under the same rule.
2. **Re-run the Arms the stopping rule cut short** (v1's configuration, shallower trees, the two-part model), with more patience or stopping on the Blend, as new pre-registered experiments.
3. **Calibrate Scenario Effects** against the PPML benchmark and a pool of Event Backtests, and report a range rather than a point.
4. **Make the Potential Range relative to Export Potential**, so that it means something for large pairs.
5. **Add sanctions and post-2021 tariffs** when a source allows, so that Russia-style Opportunities disappear and tariff Scenarios start from current rates.
6. **Try an entity-embedding network** for Exporters, Importers and Products under the same rule; the course's advice that trees win on tabular data is the hypothesis to beat (*Course:* embeddings).
7. **Name an owner**, store the lineage hashes in the artefact, and use the 2025 data as the first live test of the CPTPP Scenario.

**Conclusion.** The UK Exports Predictor is a tool for deciding *where to look*. Chosen on the measure Advisors read, it ranks UK Markets better than a market-share rule on a year it never saw, by a modest margin, and its weaknesses are measured rather than hidden: dollar values deserve less trust than rankings, and Scenario Effects are illustrative upper ends. Its numbers are associations learned from global trade, useful for prioritising conversations with industry and never a promise of what a Market or an agreement will deliver.

## Appendix: frozen results

These parts chose or checked v2 and no longer decide anything (ADR 0008). Each is one short reported result, read from the artefact or the saved experiment results, with a link to its detail. None is extended, re-run or re-tuned here, and a part this artefact version did not run says so. Each entry names the course concept it applied. They are kept because they explain how v2 came to be and why it fell short, and because the code that produced them still runs and is still tested; deleting them would hide the path that led to the Selection Measure.

### The hyperparameter search

v2's search (Optuna's TPE sampler with a median pruner; *Course:* grid and random search, taken to Bayesian optimisation) ran 27 trials on a Sector subset and retrained the best three on every Product, choosing on 2022 Tweedie deviance with the UK ranking as a guardrail. Trial 15 won: depth 12, variance power 1.27, 1,198 rounds, with the lowest deviance (364.3) and the lowest UK ranking of the three (0.528 against 0.531 and 0.536). The guardrail passed by a hair, −0.0099 against −0.01. Its rounds, scaled to 1,453, are the served round count (section 7). Detail: ADR 0002 and ADR 0008.

In [ ]:
from exportspredictor import search

search_results = bundle.metadata["search"]
winner = search_results["winner"]
print(
    f"Winner: trial {winner['trial']}, {winner['rounds']} rounds; UK ranking against v1's configuration "
    f"{search_results['spearman_vs_v1']} (guardrail {search.SPEARMAN_GUARDRAIL})"
)
{name: winner["params"][name] for name in search.SPACE}

### The feature ablation

One model with every feature, then one without each of v2's four candidate features, on one fold and a Sector subset, measured on 2022 (*Course:* drop-column importance, the retrain-without-it cousin of permutation importance; a feature with no signal adds variance and removes no bias). A feature was dropped only if the model without it was better on both deviance and the UK ranking. Market import growth was dropped; Relatedness and the re-export ratio were kept on deviance, and the Applied Tariff by rule, because tariff Scenarios need it. Every model without one feature ranked UK pairs slightly better, which looks more like one-fold noise than signal. Detail: `exportspredictor.experiments`.

In [ ]:
ablation = pl.DataFrame(experiment_results["ablation"])
print("Dropped:", [f for f in features.ALL_FEATURES if f not in experiment_results["kept_features"]] or "none")
ablation.select(
    "dropped", "features", "training_rows", "rounds", "validation_deviance", "deviance_change",
    "uk_spearman", "spearman_change", "hurts",
)

### The learning curve

The same model at non-UK sampling rates of 0.1, 0.3 and 0.65 and Zero Cell rates of 2%, 5% and 10% (*Course:* the bias–variance trade-off, read along the data axis rather than model complexity). Deviance was still falling at the highest rate, by 4–6% from 0.3 to 0.65, so the model is variance-limited on large Flows and trains on every non-UK row the VRAM budget allows; the Zero Cell rate stayed at 5%. The UK ranking moved by less than 0.01 across the grid: more rows sharpen the scale of Export Potential more than its order. Detail: ADR 0004.

In [ ]:
curve = pl.DataFrame(experiment_results["learning_curve"])
curve.select(
    "zero_rate", "non_uk_rate", "training_rows", "rounds", "validation_deviance", "uk_spearman",
    "full_data_non_uk_rate", "fits_budget",
)

### The v1 comparison

Before 2023 was read, a rule was fixed for whether a new version beats v1 (*Course:* touch the test set once, applied to the comparison itself): the primary metric's paired-bootstrap interval lies above 0, the margin over the Naive Baseline is larger than v1's, and no secondary metric is worse beyond its interval. v1 is never re-run; its frozen tables are checksummed. Under that rule v2 did not beat v1 (0.511 against 0.540). Run for the served Blend with the same code, which calls the newer version "v2", the result is **mixed**: the Blend ranks better than v1 (0.555 against 0.540, interval above 0) and beats the Naive Baseline by more, but its Tweedie deviance and size-matched Lift are worse beyond their intervals, the cost section 7 reports. Detail: `exportspredictor.comparison`.

In [ ]:
v1_comparison = report.get("v1_comparison")
print(f"The v1 comparison: {NOT_RUN}." if v1_comparison is None else v1_comparison["verdict"]["text"])

### The gain ladder

For v2, two more test fold models climbed from v1 to v2 one group of changes at a time: v1's features and configuration on v2's training rows, then v2's new features, then its tuning (*Course:* ablation run the other way, adding one group of changes at a time). On artefact `20261118T082751Z` it associated almost all of v2's drop with the first step, the new training rows (−0.020 of −0.028), and about −0.004 each with the features and the tuning; another order of steps could split it differently. It was run for v2 only, so for the served version the cell below says it was not run. Detail: ADR 0008.

In [ ]:
reporting.gain_ladder(bundle)

### The constraint comparison

Two models that differ only in the monotone constraint were asked the same Scenario questions on three Sectors and one held-out fold (*Course:* a monotone constraint as domain knowledge built into boosting, a form of regularisation). On v2 (artefact `20261118T042154Z`), without the constraint 39% of added agreements lowered Export Potential, and the total effect of adding one was +4.0% against +26.6% with it: the constraint makes the direction consistent and the size larger. The experiment loop then tested the constraints against the ranking: removing them ranked slightly worse, inside the noise, so they stay (ADR 0008). The comparison was not re-run for the served version:

In [ ]:
monotone = report.get("monotone_constraints")
print(f"Monotone-constraint comparison: {NOT_RUN}." if monotone is None else monotone["text"])

**Conclusion.** Each frozen part answered a question about v2: which features, how many rows, which configuration, whether it beat v1, where its loss came from, and what the constraint does. Each answer stands as a reported result, and none decides anything about the served Blend, which was chosen on the Selection Measure alone.